# Chapter 15 - Processing Sequences Using RNNs and CNNs

## 1. Recurrent Neurons and Sequence Types

Recurrent Neural Network (RNN) memperkenalkan **koneksi rekuren** yang memungkinkan neuron menerima tidak hanya input saat ini x(t), tetapi juga **state atau output dari langkah waktu sebelumnya** y(t−1). Dengan mekanisme ini, output pada waktu t menjadi fungsi dari seluruh urutan input sebelumnya. Ketika RNN di-*unroll* sepanjang waktu, neuron yang sama muncul berulang di setiap time step dengan **bobot yang dibagi (shared weights)**.

Secara matematis, satu layer RNN sederhana menggunakan dua matriks bobot utama: Wₓ untuk koneksi dari input dan Wₕ untuk koneksi dari hidden state sebelumnya, ditambah bias b dan fungsi aktivasi ϕ (umumnya tanh).  

Berdasarkan relasi antara input dan output, RNN dapat dikonfigurasikan ke dalam beberapa tipe utama:
- **Sequence-to-sequence**: input dan output sama-sama berupa urutan.
- **Sequence-to-vector**: memetakan urutan ke satu vektor (misalnya klasifikasi sentimen).
- **Vector-to-sequence**: memetakan satu vektor ke urutan (misalnya image captioning).
- **Encoder–decoder**: mengodekan urutan sumber ke representasi vektor lalu mendekodenya menjadi urutan target, seperti pada sistem penerjemahan mesin.


## **Example: SimpleRNN for Time Series Forecasting**

In [ ]:
import numpy as np
from tensorflow import keras

def generate_time_series(batch_size, n_steps):
    freq1, freq2, offs1, offs2 = np.random.rand(4, batch_size, 1)
    time = np.linspace(0, 1, n_steps)
    series = 0.5 * np.sin((time - offs1) * (freq1 * 10 + 10))
    series += 0.2 * np.sin((time - offs2) * (freq2 * 20 + 20))
    series += 0.1 * (np.random.rand(batch_size, n_steps) - 0.5)
    return series[..., np.newaxis].astype(np.float32)

n_steps = 50
series = generate_time_series(10000, n_steps + 1)
X_train, y_train = series[:7000, :n_steps], series[:7000, -1]
X_valid, y_valid = series[7000:9000, :n_steps], series[7000:9000, -1]

model = keras.models.Sequential([
    keras.layers.SimpleRNN(1, input_shape=[None, 1])
])
model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train, y_train, epochs=20,
                    validation_data=(X_valid, y_valid))


## 2. Training RNNs and Time Series Forecasting

RNN dilatih menggunakan **Backpropagation Through Time (BPTT)**, yaitu dengan meng-*unroll* jaringan sepanjang beberapa langkah waktu. Forward pass menghitung output dan loss pada seluruh time step, sedangkan backward pass menyebarkan gradien ke belakang melalui waktu. Karena bobot digunakan berulang, gradien untuk parameter yang sama dijumlahkan sepanjang urutan, yang membuat RNN rentan terhadap **vanishing dan exploding gradients**, terutama pada urutan panjang.

Sebagai ilustrasi, bab ini menggunakan tugas **time series forecasting**. Data sintetis dibentuk dari kombinasi dua gelombang sinus dengan noise, lalu dipecah menjadi data train, validation, dan test dengan format [batch, time_steps, 1].  

Beberapa model dibandingkan:
- baseline naïve (memprediksi nilai terakhir),
- model linier sederhana (Flatten + Dense),
- RNN dangkal,
- dan deep RNN (2–3 layer SimpleRNN dengan return_sequences=True).

Hasilnya menunjukkan bahwa **RNN yang lebih dalam dengan Dense output** mampu melampaui baseline linier dalam memprediksi nilai masa depan.


## **Example: Deep RNN and Multi‑Step Forecasting**

In [ ]:
from tensorflow import keras
import numpy as np

# One-step-ahead deep RNN with Dense output
model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(1)
])
model.compile(loss="mse", optimizer="adam")
model.fit(X_train, y_train, epochs=20,
          validation_data=(X_valid, y_valid))

# Forecast 10 steps ahead at once (sequence-to-vector)
series = generate_time_series(10000, n_steps + 10)
X_train, Y_train = series[:7000, :n_steps], series[:7000, -10:, 0]
X_valid, Y_valid = series[7000:9000, :n_steps], series[7000:9000, -10:, 0]

model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20),
    keras.layers.Dense(10)
])
model.compile(loss="mse", optimizer="adam")
model.fit(X_train, Y_train, epochs=20,
          validation_data=(X_valid, Y_valid))


## 3. Fighting Unstable Gradients and Using LSTM/GRU

RNN menghadapi dua keterbatasan utama: **gradien yang tidak stabil** dan **memori jangka pendek**. Beberapa teknik dari deep feedforward networks tetap relevan, seperti optimizer berbasis momentum, gradient clipping, dropout, dan inisialisasi bobot yang baik. Namun, **Batch Normalization** kurang efektif di dalam sel RNN karena ketergantungan waktu.

Sebagai alternatif, **Layer Normalization** lebih cocok karena menormalkan aktivasi sepanjang dimensi fitur untuk setiap instance dan time step, sehingga membantu stabilitas training.

Untuk mengatasi keterbatasan memori, diperkenalkan **Long Short-Term Memory (LSTM)** dan **Gated Recurrent Unit (GRU)**. Kedua arsitektur ini menggunakan mekanisme **gate** (forget, input, output, update/reset) serta state jangka panjang yang memungkinkan jaringan mempertahankan informasi penting selama puluhan langkah waktu. Dalam praktik, LSTM dan GRU jauh lebih mudah dilatih pada urutan panjang dibanding SimpleRNN dan sering menjadi pilihan default untuk masalah sekuens.


## **Example: LSTM / GRU and Layer‑Norm SimpleRNNCell**

In [ ]:
from tensorflow import keras

# Stacked LSTM seq2seq
model_lstm = keras.models.Sequential([
    keras.layers.LSTM(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.LSTM(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

# GRU with Conv1D front-end (see next section for context)
model_gru = keras.models.Sequential([
    keras.layers.GRU(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

# Custom SimpleRNNCell with Layer Normalization
class LNSimpleRNNCell(keras.layers.Layer):
    def __init__(self, units, activation="tanh", **kwargs):
        super().__init__(**kwargs)
        self.state_size = units
        self.output_size = units
        self.simple_rnn_cell = keras.layers.SimpleRNNCell(units, activation=None)
        self.layer_norm = keras.layers.LayerNormalization()
        self.activation = keras.activations.get(activation)

    def call(self, inputs, states):
        outputs, _ = self.simple_rnn_cell(inputs, states)
        norm_outputs = self.activation(self.layer_norm(outputs))
        return norm_outputs, [norm_outputs]

model_ln = keras.models.Sequential([
    keras.layers.RNN(LNSimpleRNNCell(20), return_sequences=True,
                     input_shape=[None, 1]),
    keras.layers.RNN(LNSimpleRNNCell(20), return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])


## 4. Mixing RNNs with 1D Convolutions and WaveNet

Untuk urutan yang sangat panjang, bab ini mengeksplorasi kombinasi **Conv1D dan RNN**, atau bahkan mengganti RNN sepenuhnya dengan CNN satu dimensi. Conv1D dengan stride > 1 atau padding *valid* dapat melakukan **downsampling temporal**, sehingga memperpendek urutan sambil tetap menangkap pola lokal.

Dengan memilih kernel size dan stride yang tepat, model dapat melatih GRU atau LSTM di atas representasi urutan yang lebih ringkas. Output multi-step kemudian dihasilkan menggunakan **TimeDistributed(Dense)**, yang terbukti meningkatkan performa forecasting.

Selain itu, diperkenalkan **WaveNet**, arsitektur berbasis CNN murni dengan **dilated convolutions** yang dilation-nya meningkat secara eksponensial (1, 2, 4, 8, …). Pendekatan ini memungkinkan model menangkap ketergantungan jangka sangat panjang secara efisien dan mencapai performa tinggi pada tugas audio dan sekuens panjang.


## **Example: Conv1D + GRU and Simplified WaveNet**

In [ ]:
from tensorflow import keras

# Conv1D front-end + GRU stack
model = keras.models.Sequential([
    keras.layers.Conv1D(
        filters=20, kernel_size=4, strides=2, padding="valid",
        input_shape=[None, 1]
    ),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.GRU(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

model.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])
history = model.fit(
    X_train, Y_train[:, 3::2], epochs=20,
    validation_data=(X_valid, Y_valid[:, 3::2])
)

# Simplified WaveNet-style dilated Conv1D stack
wavenet = keras.models.Sequential()
wavenet.add(keras.layers.InputLayer(input_shape=[None, 1]))
for rate in (1, 2, 4, 8) * 2:
    wavenet.add(keras.layers.Conv1D(
        filters=20, kernel_size=2, padding="causal",
        activation="relu", dilation_rate=rate
    ))
wavenet.add(keras.layers.Conv1D(filters=10, kernel_size=1))
wavenet.compile(loss="mse", optimizer="adam", metrics=[last_time_step_mse])
history_w = wavenet.fit(
    X_train, Y_train, epochs=20,
    validation_data=(X_valid, Y_valid)
)


## 5. Practical Forecasting Patterns and Multi-Step Strategies

Bab ini menutup dengan beberapa strategi praktis untuk **multi-step forecasting**:

- **Iterative strategy**  
  Model dilatih untuk memprediksi satu langkah ke depan, lalu prediksi tersebut digunakan kembali sebagai input untuk langkah berikutnya. Metode ini sederhana namun rentan terhadap akumulasi error.

- **Direct strategy**  
  Model dilatih untuk langsung memprediksi beberapa langkah ke depan sekaligus sebagai output vektor. Pendekatan ini lebih stabil, meskipun bisa lebih sulit untuk horizon yang sangat jauh.

- **Sequence-to-sequence multi-horizon**  
  Target diubah menjadi urutan vektor multi-step, sehingga setiap time step memprediksi beberapa langkah ke depan. Dengan RNN seq-to-seq dan TimeDistributed(Dense), sinyal loss menjadi lebih kaya, yang sering mempercepat dan menstabilkan training.

Strategi-strategi ini dikombinasikan dengan baseline yang kuat, pilihan arsitektur (SimpleRNN, LSTM, GRU, Conv1D, WaveNet), serta teknik stabilisasi seperti clipping, normalisasi, dan dropout untuk membangun model forecasting yang benar-benar kompetitif.

## **Example: Sequence‑to‑Sequence Multi‑Horizon Targets**

In [ ]:
import numpy as np
from tensorflow import keras

n_steps = 50
series = generate_time_series(10000, n_steps + 10)

Y = np.empty((10000, n_steps, 10))
for step_ahead in range(1, 11):
    Y[:, :, step_ahead - 1] = series[:, step_ahead: step_ahead + n_steps, 0]

Y_train, Y_valid, Y_test = Y[:7000], Y[7000:9000], Y[9000:]

model = keras.models.Sequential([
    keras.layers.SimpleRNN(20, return_sequences=True, input_shape=[None, 1]),
    keras.layers.SimpleRNN(20, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(10))
])

model.compile(loss="mse", optimizer="adam")
history = model.fit(X_train, Y_train, epochs=20,
                    validation_data=(X_valid, Y_valid))
